In [1]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.5 MB/s eta 0:00:00a 0:00:01


In [2]:
from ultralytics import YOLO
import os
import glob
import yaml


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
import os

if os.path.exists("/kaggle/input/dsai-unified-dataset"):
    DATA_PATH = "/kaggle/input/dsai-unified-dataset"
else:
    DATA_PATH = "/kaggle/input/datasets/ds22f1001123/dsai-unified-dataset"

print("Using dataset path:", DATA_PATH)

Using dataset path: /kaggle/input/datasets/ds22f1001123/dsai-unified-dataset


In [11]:
with open("data.yaml", "w") as f:
    f.write(f"""
path: {DATA_PATH}
train: images/train
val: images/val

# Point to the new labels specifically
test: /kaggle/working/collapsed_labels/val 
train_labels: /kaggle/working/collapsed_labels/train
val_labels: /kaggle/working/collapsed_labels/val

names:
  0: person
  1: door
  2: obstacle
""")

In [10]:
import os
from tqdm import tqdm

# Paths from your notebook
INPUT_LABELS = os.path.join(DATA_PATH, "labels")
OUTPUT_BASE = "/kaggle/working/collapsed_labels"

# Mapping: {original_id: new_id}
# person (0) stays 0, door (6) becomes 1, everything else becomes 2
class_mapping = {0: 0, 6: 1} 

def collapse_labels(split):
    input_dir = os.path.join(INPUT_LABELS, split)
    output_dir = os.path.join(OUTPUT_BASE, split)
    os.makedirs(output_dir, exist_ok=True)
    
    label_files = [f for f in os.listdir(input_dir) if f.endswith('.txt')]
    
    for filename in tqdm(label_files, desc=f"Processing {split}"):
        with open(os.path.join(input_dir, filename), 'r') as f:
            lines = f.readlines()
        
        new_lines = []
        for line in lines:
            parts = line.split()
            if not parts: continue
            
            old_cls = int(parts[0])
            # If it's person or door, use mapping; otherwise, it's an obstacle (2)
            new_cls = class_mapping.get(old_cls, 2)
            
            # Reconstruct the line: new_class x y w h
            new_lines.append(f"{new_cls} {' '.join(parts[1:])}\n")
            
        with open(os.path.join(output_dir, filename), 'w') as f:
            f.writelines(new_lines)

# Process both train and val splits
collapse_labels("train")
collapse_labels("val")

Processing val: 100%|██████████| 2340/2340 [00:13<00:00, 176.07it/s]


In [12]:
model = YOLO("yolo11s.pt")
DATA_YAML = "/kaggle/working/data.yaml"

In [13]:
# with open("YAML.yaml", "w") as f:
#     f.write("""
# path: /kaggle/input/datasets/ds22f1001123/dsai-unified-dataset

# train: images/train
# val: images/val

# names:
#   0: person
#   1: table
#   2: potted plant
#   3: chair
#   4: sofa
#   5: lamp
#   6: door
#   7: cabinet
#   8: wardrobe
#   9: refrigerator
#   10: bed
#     """)

In [ ]:
results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=32,
    device=0,

    single_cls=False,

    # 🔥 learning
    lr0=0.005,
    lrf=0.01,
    weight_decay=0.0007,

    # 🔥 augmentations (slightly reduced)
    degrees=6.0,
    translate=0.08,
    scale=0.3,
    fliplr=0.5,
    hsv_h=0.01,
    hsv_s=0.4,
    hsv_v=0.3,

    # 🔥 spatial robustness
    mosaic=1.0,
    close_mosaic=15,
    mixup=0.05,

    # 🔥 stability
    patience=30,
    cache=True
)

In [ ]:
val_model = YOLO('/kaggle/working/runs/detect/train/weights/best.pt')

metrics = val_model.val(data=DATA_YAML)

print(f"mAP@50-95: {metrics.box.map:.4f}")
print(f"mAP@50:    {metrics.box.map50:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

results_dir = '/kaggle/working/runs/detect/train/'

plt.figure(figsize=(10, 10))
img = mpimg.imread(os.path.join(results_dir, 'confusion_matrix.png'))
plt.imshow(img)
plt.axis('off')
plt.title('Confusion Matrix')
plt.show()

plt.figure(figsize=(12, 8))
img = mpimg.imread(os.path.join(results_dir, 'results.png'))
plt.imshow(img)
plt.axis('off')
plt.title('Training Metrics and Loss Curves')
plt.show()